# 6. Regression Analysis

Dự đoán `freight_value` cho từng order item tại checkout bằng đúng hai thuật toán: Linear Regression và Random Forest. Validation 07/2018 dùng để chọn model; test 08/2018 chỉ đánh giá model đã chọn.


## 1. Đọc và kiểm tra dữ liệu

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

path = Path('../data/cleaned/olist_clean.csv')
if not path.exists():
    path = Path('data/cleaned/olist_clean.csv')

df = pd.read_csv(path)
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

print(df.shape)
print('Missing:', df.isnull().sum().sum())
print('Trùng grain:', df.duplicated(['order_id', 'order_item_id']).sum())

(108937, 27)
Missing: 0
Trùng grain: 0


Kết quả đọc file phải khớp Cleaning: 108.937 item, 27 cột, không thiếu và không trùng grain (`order_id`, `order_item_id`). Target là `freight_value` theo item, đơn vị BRL.


## 2. Chọn feature và target

In [2]:
numeric_features = [
    'price', 'weight_g', 'volume_cm3',
    'distance_km', 'order_n_items', 'same_state'
]

category_features = [
    'category', 'customer_state', 'seller_state'
]

features = numeric_features + category_features
target = 'freight_value'

print('Features:', features)
print('Target:', target)

Features: ['price', 'weight_g', 'volume_cm3', 'distance_km', 'order_n_items', 'same_state', 'category', 'customer_state', 'seller_state']
Target: freight_value


Chín feature tạo thành ba nhóm đã được xác định từ bài toán và dữ liệu có tại checkout: đặc điểm vật lý/sản phẩm (`price`, `weight_g`, `volume_cm3`, `category`), quy mô order (`order_n_items`) và địa lý (`distance_km`, `same_state`, `customer_state`, `seller_state`). EDA cho thấy mỗi nhóm có tín hiệu khác nhau nhưng không feature nào giải thích toàn bộ freight.

`order_n_items` vẫn được giữ dù Pearson gần 0 vì Pearson chỉ đo tuyến tính và các nhóm order có phân phối khác nhau. Weight và volume cùng được giữ dù tương quan 0,81 vì chúng đo hai khía cạnh vật lý khác nhau; validation sẽ đánh giá tổ hợp thay vì loại trước một biến. ID bị loại vì chỉ định danh và có cardinality cao; payment chứa trực tiếp target, còn review và kết quả giao hàng chỉ có sau checkout. Thời gian dùng để chia tập, không dùng làm feature chính.


## 3. Chia train, validation và test theo thời gian

In [3]:
train = df[df['order_purchase_timestamp'] < '2018-07-01'].copy()
validation = df[
    (df['order_purchase_timestamp'] >= '2018-07-01') &
    (df['order_purchase_timestamp'] < '2018-08-01')
].copy()
test = df[
    (df['order_purchase_timestamp'] >= '2018-08-01') &
    (df['order_purchase_timestamp'] < '2018-09-01')
].copy()

print('Train:', train.shape)
print('Validation:', validation.shape)
print('Test:', test.shape)

print('Train - Validation trùng order:', len(set(train['order_id']) & set(validation['order_id'])))
print('Train - Test trùng order:', len(set(train['order_id']) & set(test['order_id'])))
print('Validation - Test trùng order:', len(set(validation['order_id']) & set(test['order_id'])))

Train: (94898, 27)
Validation: (6934, 27)
Test: (7105, 27)
Train - Validation trùng order: 0
Train - Test trùng order: 0
Validation - Test trùng order: 0


Validation dùng để chọn mô hình. Test chỉ được sử dụng sau khi đã chọn xong mô hình.

## 4. Chuẩn bị X và y

In [4]:
X_train = train[features].copy()
X_val = validation[features].copy()

y_train = train[target]
y_val = validation[target]

X_train = pd.get_dummies(X_train, columns=category_features, dtype=int)
X_val = pd.get_dummies(X_val, columns=category_features, dtype=int)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print('X_train:', X_train.shape)
print('X_val:', X_val.shape)

X_train:

 (94898, 129)
X_val: (6934, 129)


`get_dummies()` mã hóa các biến phân loại. Validation được đưa về đúng các cột của train nên không dùng category của validation để tạo cột mới cho mô hình.

## 5. Linear Regression


In [5]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_val)

linear_mae = mean_absolute_error(y_val, linear_pred)
linear_rmse = mean_squared_error(y_val, linear_pred) ** 0.5
linear_r2 = r2_score(y_val, linear_pred)

print('Linear MAE:', round(linear_mae, 3))
print('Linear RMSE:', round(linear_rmse, 3))
print('Linear R²:', round(linear_r2, 3))
print('Dự đoán âm:', (linear_pred < 0).sum())

Linear MAE: 6.068
Linear RMSE: 13.856
Linear R²: 0.572
Dự đoán âm: 0


In [6]:
linear_coefficients = pd.Series(linear_model.coef_, index=X_train.columns)
display(linear_coefficients[numeric_features].to_frame('Coefficient').round(4))

,Coefficient
price,0.0145
weight_g,0.0014
volume_cm3,0.0002
distance_km,0.0064
order_n_items,-0.1143
same_state,-3.7251


Linear Regression học một tổ hợp tuyến tính: intercept cộng từng giá trị đầu vào nhân hệ số. Với các feature khác giữ nguyên, hệ số số cho biết dự đoán thay đổi bao nhiêu BRL khi feature tăng một đơn vị; đây không phải tác động nhân quả. Bảng chỉ hiển thị sáu hệ số số vì các cột one-hot category/state được giữ đầy đủ cùng intercept, nên từng hệ số phân loại không có cách diễn giải độc lập ổn định. Weight và volume tương quan 0,81 cũng làm hệ số riêng nhạy với feature còn lại.

Trên validation, Linear Regression đạt MAE 6,068 BRL, RMSE 13,856 BRL và R² 0,572. Model giải thích được một phần đáng kể biến thiên, nhưng sai số còn cao cho thấy tổ hợp tuyến tính chưa biểu diễn hết độ phân tán, quan hệ phi tuyến và tương tác quan sát trong EDA.


## 6. Random Forest


Random Forest được thử sau Linear Regression vì nhiều cây quyết định có thể chia dữ liệu theo các ngưỡng khác nhau, học quan hệ phi tuyến và tương tác, rồi lấy trung bình dự đoán của các cây. Điều này phù hợp để kiểm tra các mẫu mà một đường tuyến tính khó biểu diễn, nhưng không có nghĩa model đã học đúng cơ chế tính cước thực tế.

`n_estimators=100` tạo 100 cây để lấy trung bình; `max_depth=18` và `min_samples_leaf=3` giới hạn cây/quy định số mẫu tối thiểu ở lá; `max_features=0.8` cho mỗi lần chia chỉ xét một phần feature; `random_state=42` giúp tái lập và `n_jobs=-1` chỉ tăng tốc. Đây là cấu hình thực nghiệm cố định, không phải kết quả tối ưu hyperparameter và không được điều chỉnh theo test.


In [7]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_val)

rf_mae = mean_absolute_error(y_val, rf_pred)
rf_rmse = mean_squared_error(y_val, rf_pred) ** 0.5
rf_r2 = r2_score(y_val, rf_pred)

print('Random Forest MAE:', round(rf_mae, 3))
print('Random Forest RMSE:', round(rf_rmse, 3))
print('Random Forest R²:', round(rf_r2, 3))
print('Dự đoán âm:', (rf_pred < 0).sum())

Random Forest MAE: 5.087
Random Forest RMSE: 11.882
Random Forest R²: 0.685
Dự đoán âm: 0


## 7. So sánh hai model trên validation


In [8]:
validation_result = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [linear_mae, rf_mae],
    'RMSE': [linear_rmse, rf_rmse],
    'R²': [linear_r2, rf_r2]
})

validation_result = validation_result.sort_values('MAE')
display(validation_result.round(3))


,Model,MAE,RMSE,R²
1,Random Forest,5.087,11.882,0.685
0,Linear Regression,6.068,13.856,0.572


Random Forest có validation MAE 5,087 BRL, thấp hơn Linear Regression 0,981 BRL (khoảng 16,2%). RMSE cũng giảm từ 13,856 xuống 11,882 BRL và R² tăng từ 0,572 lên 0,685, nên cả ba metric cùng hỗ trợ lựa chọn Random Forest chứ không chỉ MAE. Quyết định này chỉ dùng validation; test chưa được xem ở bước chọn model. Đánh đổi là Random Forest khó giải thích từng dự đoán hơn các hệ số của Linear Regression.


## 8. Train lại model đã chọn và đánh giá test


In [9]:
train_val = pd.concat([train, validation], ignore_index=True)

X_train_val = train_val[features].copy()
X_test = test[features].copy()
y_train_val = train_val[target]
y_test = test[target]

X_train_val = pd.get_dummies(X_train_val, columns=category_features, dtype=int)
X_test = pd.get_dummies(X_test, columns=category_features, dtype=int)
X_test = X_test.reindex(columns=X_train_val.columns, fill_value=0)

final_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train_val, y_train_val)
y_pred = final_model.predict(X_test)

final_mae = mean_absolute_error(y_test, y_pred)
final_rmse = mean_squared_error(y_test, y_pred) ** 0.5
final_r2 = r2_score(y_test, y_pred)

print('Test MAE:', round(final_mae, 3))
print('Test RMSE:', round(final_rmse, 3))
print('Test R²:', round(final_r2, 3))
print('Dự đoán âm:', (y_pred < 0).sum())


Test MAE: 4.386
Test RMSE: 9.649
Test R²: 0.687
Dự đoán âm: 0


In [10]:
importance = pd.Series(final_model.feature_importances_, index=X_train_val.columns)

importance_grouped = pd.Series({
    'price': importance['price'],
    'weight_g': importance['weight_g'],
    'volume_cm3': importance['volume_cm3'],
    'distance_km': importance['distance_km'],
    'order_n_items': importance['order_n_items'],
    'same_state': importance['same_state'],
    'category': importance[importance.index.str.startswith('category_')].sum(),
    'customer_state': importance[importance.index.str.startswith('customer_state_')].sum(),
    'seller_state': importance[importance.index.str.startswith('seller_state_')].sum()
}).sort_values(ascending=False)

display(importance_grouped.to_frame('Importance').round(4))

,Importance
volume_cm3,0.3031
weight_g,0.2432
distance_km,0.1874
same_state,0.0946
price,0.0937
category,0.0257
customer_state,0.0236
seller_state,0.0200
order_n_items,0.0087


Importance của `final_model` cho thấy volume (0,3031), weight (0,2432) và distance (0,1874) được các cây sử dụng nhiều nhất; kết quả này phù hợp với EDA ở chỗ đặc điểm vật lý và khoảng cách đều chứa tín hiệu đáng chú ý. Tuy nhiên, Pearson của EDA đo liên hệ tuyến tính lịch sử, còn importance đo mức feature được dùng để giảm sai số trong các cây; hai loại bằng chứng không có cùng ý nghĩa.

Importance của các cột one-hot đã được cộng về đúng chín feature gốc. Bảng không chứng minh quan hệ nhân quả, không nói một feature làm freight tăng/giảm bao nhiêu và không giải thích chính xác một dự đoán riêng lẻ.


## 9. Phân tích sai số


In [11]:
error_df = test[
    ['freight_value', 'weight_g', 'distance_km',
     'same_state', 'order_n_items', 'category']
].copy()

error_df['prediction'] = y_pred
error_df['error'] = error_df['prediction'] - error_df['freight_value']
error_df['absolute_error'] = error_df['error'].abs()

q50 = train_val['freight_value'].quantile(0.50)
q75 = train_val['freight_value'].quantile(0.75)
q90 = train_val['freight_value'].quantile(0.90)

error_df['freight_group'] = pd.cut(
    error_df['freight_value'],
    [-np.inf, q50, q75, q90, np.inf],
    labels=['<= P50', 'P50 - P75', 'P75 - P90', '> P90']
)

freight_error = error_df.groupby('freight_group', observed=True).agg(
    Count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean'),
    Bias=('error', 'mean')
).round(2)

display(freight_error)

,Count,MAE,Bias
freight_group,,,
<= P50,3274,1.61,0.85
P50 - P75,1798,2.42,-1.30
P75 - P90,1227,4.59,-2.37
> P90,806,19.74,-18.17


In [12]:
d1 = train_val['distance_km'].quantile(0.25)
d2 = train_val['distance_km'].quantile(0.50)
d3 = train_val['distance_km'].quantile(0.75)

error_df['distance_group'] = pd.cut(
    error_df['distance_km'],
    [-np.inf, d1, d2, d3, np.inf],
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

distance_error = error_df.groupby('distance_group', observed=True).agg(
    Count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean'),
    Bias=('error', 'mean')
).round(2)

display(distance_error)

state_error = error_df.groupby('same_state').agg(
    Count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean'),
    Bias=('error', 'mean')
).round(2)

display(state_error)

,Count,MAE,Bias
distance_group,,,
Q1,2390,2.10,-0.39
Q2,1701,3.53,-2.30
Q3,1567,5.19,-3.60
Q4,1447,8.29,-4.57


,Count,MAE,Bias
same_state,,,
0,3901,6.19,-3.78
1,3204,2.19,-0.74


In [13]:
w1 = train_val['weight_g'].quantile(0.25)
w2 = train_val['weight_g'].quantile(0.50)
w3 = train_val['weight_g'].quantile(0.75)

error_df['weight_group'] = pd.cut(
    error_df['weight_g'],
    [-np.inf, w1, w2, w3, np.inf],
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

weight_error = error_df.groupby('weight_group', observed=True).agg(
    Count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean'),
    Bias=('error', 'mean')
).round(2)

display(weight_error)

order_error = error_df.groupby('order_n_items').agg(
    Count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean'),
    Bias=('error', 'mean')
).round(2)

display(order_error[order_error['Count'] >= 50])

category_error = error_df.groupby('category').agg(
    Count=('absolute_error', 'size'),
    MAE=('absolute_error', 'mean'),
    Bias=('error', 'mean')
)

category_error = category_error[category_error['Count'] >= 100]
category_error = category_error.sort_values('MAE', ascending=False).head(10)
display(category_error.round(2))

,Count,MAE,Bias
weight_group,,,
Q1,2209,2.30,-0.89
Q2,1919,2.82,-1.05
Q3,1551,3.91,-2.40
Q4,1426,10.25,-6.59


,Count,MAE,Bias
order_n_items,,,
1,5751,4.13,-2.78
2,881,4.85,-1.87
3,225,5.91,-1.57
4,88,6.39,-1.48
5,65,5.57,2.72
6,78,8.29,7.47


,Count,MAE,Bias
category,,,
garden_tools,136,8.68,-5.26
furniture_decor,428,6.42,-3.47
construction_tools_construction,162,6.04,-0.30
baby,212,5.41,-3.69
toys,163,5.31,-2.61
housewares,614,5.05,-3.30
cool_stuff,116,5.01,-2.12
bed_bath_table,650,4.70,-3.40
stationery,168,4.40,-2.68


### Kết luận Error Analysis

Random Forest cuối đạt MAE 4,386 BRL, RMSE 9,649 BRL và R² 0,687 trên test. Model dự đoán tốt hơn ở các vùng phổ biến: nhóm cùng bang có MAE 2,19 BRL, nhóm khoảng cách Q1 là 2,10 BRL và nhóm cân nặng Q1 là 2,30 BRL. Sai số tăng ở nhóm khác bang (6,19 BRL), khoảng cách Q4 (8,29 BRL) và cân nặng Q4 (10,25 BRL); trong các category có ít nhất 100 mẫu test, `garden_tools` cao nhất với MAE 8,68 BRL.

Vấn đề lớn nhất là nhóm freight trên P90: MAE 19,74 BRL và bias -18,17 BRL cho thấy model thường dự đoán thiếu những mức phí rất cao. Một giả thuyết hợp lý là vùng đuôi có ít quan sát hơn và trung bình dự đoán giữa các cây kéo giá trị về vùng phổ biến, nhưng dữ liệu hiện tại chưa chứng minh đây là nguyên nhân duy nhất. Nhóm P90 được xác định bằng target thật chỉ để hậu kiểm test; không thể dùng điều kiện này làm cảnh báo đầu vào cho order mới.

## 10. Minh họa sản phẩm cuối

Phần dưới minh họa cách dùng `final_model` trên một order item đã chuẩn bị đủ chín feature. Đây là demo trong notebook, chưa phải ứng dụng nhập liệu hoàn chỉnh.


In [14]:
example_input = test[features].iloc[[0]].copy()
example_encoded = pd.get_dummies(example_input, columns=category_features, dtype=int)
example_encoded = example_encoded.reindex(columns=X_train_val.columns, fill_value=0)

example_prediction = final_model.predict(example_encoded)[0]
example_actual = y_test.iloc[0]
example_error = abs(example_prediction - example_actual)

display(example_input)
print('Freight dự đoán:', round(example_prediction, 2), 'BRL')
print('Freight thực tế:', round(example_actual, 2), 'BRL')
print('Sai số tuyệt đối:', round(example_error, 2), 'BRL')

,price,weight_g,volume_cm3,distance_km,order_n_items,same_state,category,customer_state,seller_state
3,12.99,200.0,2400.0,295.03248,1,1,perfumery,SP,SP


Freight dự đoán: 12.05 BRL
Freight thực tế: 12.79 BRL
Sai số tuyệt đối: 0.74 BRL


Đây là demo dự đoán trên một dòng dữ liệu **đã được chuẩn bị**, chưa phải ứng dụng nhập liệu hoàn chỉnh. Người dùng cần đủ chín feature; ba biến phân loại được `get_dummies()` rồi `reindex()` theo đúng cột train + validation. `example_input` không chứa `freight_value`; model chỉ trả về freight dự đoán bằng BRL. Freight thực tế của item test chỉ dùng để tính sai số đối chiếu.

Trong ví dụ, item `perfumery` có giá 12,99 BRL, nặng 200 g, thể tích 2.400 cm³, khoảng cách 295 km, một item trong order và giao cùng bang SP. Model dự đoán 12,05 BRL; thực tế 12,79 BRL, sai số tuyệt đối 0,74 BRL. Với order mới, ZIP/tọa độ phải được chuẩn bị thành `customer_state`, `seller_state`, `same_state` và `distance_km` trước khi gọi model; khi chưa có freight thật, output chỉ được dùng như mức phí tham khảo và chưa thể tính sai số. Cần thận trọng hơn với hàng nặng, tuyến xa, trường hợp có thể thuộc vùng phí rất cao, freight bằng 0 hoặc dữ liệu ngoài phạm vi 2017–2018; không thể coi các feature đáng chú ý là nguyên nhân chính xác của một dự đoán riêng lẻ.


## 11. Kết luận cuối

Nhóm đã xây dựng quy trình dự đoán `freight_value` cho từng order item từ chín feature có thể chuẩn bị tại checkout trên dữ liệu Olist lịch sử. Trong hai phương pháp được đề xuất từ EDA, Random Forest được chọn hoàn toàn bằng validation vì MAE 5,087 BRL thấp hơn Linear Regression 6,068 BRL, đồng thời RMSE thấp hơn và R² cao hơn. Sau khi fit lại trên train + validation, model đạt MAE 4,386 BRL, RMSE 9,649 BRL và R² 0,687 trên test tháng 08/2018.

Model hoạt động tốt hơn với hàng nhẹ, khoảng cách ngắn và giao cùng bang, nhưng khó hơn ở hàng nặng, tuyến xa và vùng freight rất cao. Đặc biệt, nhóm trên P90 có MAE 19,74 BRL và bias -18,17 BRL, nên mức phí cao thường bị đánh giá thấp. Kết quả có thể hỗ trợ người bán tham khảo và phân tích logistics, không phải báo giá chính thức.

Phạm vi áp dụng chỉ gồm order delivered, freight dương trong giai đoạn 2017–2018. Khoảng cách là Haversine từ ZIP đại diện và dữ liệu thiếu carrier, dịch vụ, biểu phí cùng khuyến mãi; vì vậy công cụ không thay thế hệ thống tính cước thực tế và cần được kiểm chứng lại trước khi dùng cho thời gian hoặc thị trường khác.